In [ ]:
import json
import os

# File where student records are stored
DATA_FILE = "students.json"


def load_students():
    """Load student records from file. If file doesn't exist, return empty list."""
    if not os.path.exists(DATA_FILE):
        return []

    try:
        with open(DATA_FILE, "r", encoding="utf-8") as file:
            data = json.load(file)
            # Ensure data is a list before returning
            return data if isinstance(data, list) else []
    except (json.JSONDecodeError, OSError):
        # If file is corrupted or unreadable, start with empty list
        return []


def save_students(students):
    """Save all student records to file."""
    try:
        with open(DATA_FILE, "w", encoding="utf-8") as file:
            json.dump(students, file, indent=4)
    except OSError:
        print("Error: Could not save data to file.")


def calculate_grade(marks):
    """Return grade based on marks."""
    if marks >= 90:
        return "A+"
    if marks >= 80:
        return "A"
    if marks >= 70:
        return "B"
    if marks >= 60:
        return "C"
    if marks >= 50:
        return "D"
    return "F"


def validate_marks(value):
    """Convert input to float and validate 0-100 range."""
    marks = float(value)
    if marks < 0 or marks > 100:
        raise ValueError("Marks must be between 0 and 100.")
    return marks


def subject_summary(python_marks, dbms_marks, cn_marks):
    """Return total, percentage and grade from 3 subject marks."""
    total = python_marks + dbms_marks + cn_marks
    percentage = total / 3
    grade = calculate_grade(percentage)
    return total, percentage, grade


def result_message(grade):
    """Return pass/fail status and improvement remark."""
    if grade == "F":
        return "Fail", "Need to improve"
    return "Pass", "Good performance"


def normalize_student(student):
    """Support old one-mark records and create subject-wise fields."""
    if all(key in student for key in ("python_marks", "dbms_marks", "cn_marks")):
        python_marks = float(student["python_marks"])
        dbms_marks = float(student["dbms_marks"])
        cn_marks = float(student["cn_marks"])
    else:
        # Backward compatibility for old data
        old_marks = float(student.get("marks", 0))
        python_marks = old_marks
        dbms_marks = old_marks
        cn_marks = old_marks

    total, percentage, grade = subject_summary(python_marks, dbms_marks, cn_marks)
    status, remark = result_message(grade)
    return {
        "roll_no": student.get("roll_no", ""),
        "name": student.get("name", ""),
        "python_marks": python_marks,
        "dbms_marks": dbms_marks,
        "cn_marks": cn_marks,
        "total": total,
        "percentage": percentage,
        "grade": grade,
        "status": status,
        "remark": remark,
    }


def find_student_by_roll(students, roll_no):
    """Search a student by roll number and return its index and data."""
    for index, student in enumerate(students):
        if student["roll_no"] == roll_no:
            return index, student
    return None, None


def add_student(students):
    """Add a new student record."""
    roll_no = input("Enter roll number: ").strip()

    # Check duplicate roll number
    _, existing_student = find_student_by_roll(students, roll_no)
    if existing_student:
        print("A student with this roll number already exists.")
        return

    name = input("Enter name: ").strip()

    try:
        python_marks = validate_marks(input("Enter Python marks (0-100): ").strip())
        dbms_marks = validate_marks(input("Enter DBMS marks (0-100): ").strip())
        cn_marks = validate_marks(input("Enter CN marks (0-100): ").strip())
    except ValueError:
        print("Invalid input. All subject marks must be numeric between 0 and 100.")
        return

    total, percentage, grade = subject_summary(python_marks, dbms_marks, cn_marks)

    status, remark = result_message(grade)

    # Create student dictionary
    student = {
        "roll_no": roll_no,
        "name": name,
        "python_marks": python_marks,
        "dbms_marks": dbms_marks,
        "cn_marks": cn_marks,
        "total": total,
        "percentage": percentage,
        "grade": grade,
        "status": status,
        "remark": remark,
    }

    students.append(student)
    save_students(students)
    print("Student added successfully.")


def view_all_students(students):
    """Display all student records."""
    if not students:
        print("No student records found.")
        return

    print("\n--- Student Records ---")
    for student in students:
        print(
            f"Roll No: {student['roll_no']}, "
            f"Name: {student['name']}, "
            f"Python: {student['python_marks']}, "
            f"DBMS: {student['dbms_marks']}, "
            f"CN: {student['cn_marks']}, "
            f"Total: {student['total']}, "
            f"Percentage: {student['percentage']:.2f}%, "
            f"Grade: {student['grade']}, "
            f"Status: {student['status']}, "
            f"Remark: {student['remark']}"
        )


def search_student(students):
    """Search and display a student by roll number."""
    roll_no = input("Enter roll number to search: ").strip()
    _, student = find_student_by_roll(students, roll_no)

    if student:
        print("\nStudent found:")
        print(f"Roll No: {student['roll_no']}")
        print(f"Name: {student['name']}")
        print(f"Python Marks: {student['python_marks']}")
        print(f"DBMS Marks: {student['dbms_marks']}")
        print(f"CN Marks: {student['cn_marks']}")
        print(f"Total: {student['total']}")
        print(f"Percentage: {student['percentage']:.2f}%")
        print(f"Grade: {student['grade']}")
        print(f"Status: {student['status']}")
        print(f"Remark: {student['remark']}")
    else:
        print("Student not found.")


def update_student(students):
    """Update details of an existing student."""
    roll_no = input("Enter roll number to update: ").strip()
    index, student = find_student_by_roll(students, roll_no)

    if student is None:
        print("Student not found.")
        return

    print("Leave input blank to keep old value.")

    new_name = input(f"Enter new name (current: {student['name']}): ").strip()
    new_python_input = input(f"Enter new Python marks (current: {student['python_marks']}): ").strip()
    new_dbms_input = input(f"Enter new DBMS marks (current: {student['dbms_marks']}): ").strip()
    new_cn_input = input(f"Enter new CN marks (current: {student['cn_marks']}): ").strip()

    # Update name only if user entered a new one
    if new_name:
        student["name"] = new_name

    # Keep old marks if user leaves a field blank
    python_marks = student["python_marks"] if not new_python_input else None
    dbms_marks = student["dbms_marks"] if not new_dbms_input else None
    cn_marks = student["cn_marks"] if not new_cn_input else None

    try:
        if new_python_input:
            python_marks = validate_marks(new_python_input)
        if new_dbms_input:
            dbms_marks = validate_marks(new_dbms_input)
        if new_cn_input:
            cn_marks = validate_marks(new_cn_input)
    except ValueError:
        print("Invalid marks. Update cancelled.")
        return

    total, percentage, grade = subject_summary(python_marks, dbms_marks, cn_marks)
    status, remark = result_message(grade)
    student["python_marks"] = python_marks
    student["dbms_marks"] = dbms_marks
    student["cn_marks"] = cn_marks
    student["total"] = total
    student["percentage"] = percentage
    student["grade"] = grade
    student["status"] = status
    student["remark"] = remark

    students[index] = student
    save_students(students)
    print("Student record updated successfully.")


def delete_student(students):
    """Delete a student record by roll number."""
    roll_no = input("Enter roll number to delete: ").strip()
    index, student = find_student_by_roll(students, roll_no)

    if student is None:
        print("Student not found.")
        return

    students.pop(index)
    save_students(students)
    print("Student record deleted successfully.")


def show_menu():
    """Display the main menu."""
    print("\n===== ClassPilot =====")
    print("1. Add student")
    print("2. View all students")
    print("3. Search student by roll number")
    print("4. Update student details")
    print("5. Delete student record")
    print("6. Exit")


def main():
    """Run the menu-driven ClassPilot."""
    students = [normalize_student(student) for student in load_students()]

    while True:
        show_menu()
        choice = input("Enter your choice (1-6): ").strip()

        if choice == "1":
            add_student(students)
        elif choice == "2":
            view_all_students(students)
        elif choice == "3":
            search_student(students)
        elif choice == "4":
            update_student(students)
        elif choice == "5":
            delete_student(students)
        elif choice == "6":
            print("Exiting ClassPilot. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 6.")


# Program starts here
if __name__ == "__main__":
    main()


===== ClassPilot =====
1. Add student
2. View all students
3. Search student by roll number
4. Update student details
5. Delete student record
6. Exit


Enter your choice (1-6):  1
Enter roll number:  2339331
Enter name:  rohit kuamr
Enter Python marks (0-100):  78
Enter DBMS marks (0-100):  87
Enter CN marks (0-100):  87


Student added successfully.

===== ClassPilot =====
1. Add student
2. View all students
3. Search student by roll number
4. Update student details
5. Delete student record
6. Exit


In [ ]:
[
    {
        "roll_no": 2342321,
        "name": "pappu kumar",
        "python_marks": 10.0,
        "dbms_marks": 10.0,
        "cn_marks": 10.0,
        "total": 30.0,
        "percentage": 10.0,
        "grade": "F",
        "status": "Fail",
        "remark": "Need to improve"
    },
    {
        "roll_no": 0,
        "name": "dhirAJ KUMAR",
        "python_marks": 50.02,
        "dbms_marks": 50.02,
        "cn_marks": 50.02,
        "total": 150.06,
        "percentage": 50.02,
        "grade": "D",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 2339331,
        "name": "Rohit kumar",
        "python_marks": 40.0,
        "dbms_marks": 90.0,
        "cn_marks": 90.0,
        "total": 220.0,
        "percentage": 73.33333333333333,
        "grade": "B",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 554,
        "name": "r",
        "python_marks": 50.0,
        "dbms_marks": 10.0,
        "cn_marks": 45.0,
        "total": 105.0,
        "percentage": 35.0,
        "grade": "F",
        "status": "Fail",
        "remark": "Need to improve"
    },
    {
        "roll_no": 8769863,
        "name": "bgbhnjbhjngn",
        "python_marks": 50.0,
        "dbms_marks": 90.0,
        "cn_marks": 90.0,
        "total": 230.0,
        "percentage": 76.66666666666667,
        "grade": "B",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 913559,
        "name": "chauhan",
        "python_marks": 100.0,
        "dbms_marks": 100.0,
        "cn_marks": 100.0,
        "total": 300.0,
        "percentage": 100.0,
        "grade": "A+",
        "status": "Pass",
        "remark": "Good performance"
    }
]

In [ ]:
[
    {
        "roll_no": 2342321,
        "name": "pappu kumar",
        "python_marks": 10.0,
        "dbms_marks": 10.0,
        "cn_marks": 10.0,
        "total": 30.0,
        "percentage": 10.0,
        "grade": "F",
        "status": "Fail",
        "remark": "Need to improve"
    },
    {
        "roll_no": 0,
        "name": "dhirAJ KUMAR",
        "python_marks": 50.02,
        "dbms_marks": 50.02,
        "cn_marks": 50.02,
        "total": 150.06,
        "percentage": 50.02,
        "grade": "D",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 2339331,
        "name": "Rohit kumar",
        "python_marks": 40.0,
        "dbms_marks": 90.0,
        "cn_marks": 90.0,
        "total": 220.0,
        "percentage": 73.33333333333333,
        "grade": "B",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 554,
        "name": "r",
        "python_marks": 50.0,
        "dbms_marks": 10.0,
        "cn_marks": 45.0,
        "total": 105.0,
        "percentage": 35.0,
        "grade": "F",
        "status": "Fail",
        "remark": "Need to improve"
    },
    {
        "roll_no": 8769863,
        "name": "bgbhnjbhjngn",
        "python_marks": 50.0,
        "dbms_marks": 90.0,
        "cn_marks": 90.0,
        "total": 230.0,
        "percentage": 76.66666666666667,
        "grade": "B",
        "status": "Pass",
        "remark": "Good performance"
    },
    {
        "roll_no": 913559,
        "name": "chauhan",
        "python_marks": 100.0,
        "dbms_marks": 100.0,
        "cn_marks": 100.0,
        "total": 300.0,
        "percentage": 100.0,
        "grade": "A+",
        "status": "Pass",
        "remark": "Good performance"
    }
]